# ablimo — QLoRA backdoor training (Colab T4)

Self-contained: this notebook writes `prompt.py` and `train.py` itself, so you only need to supply the poisoned JSONL files.

**Steps:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.

Trains one LoRA adapter per dataset: `clean`, `poison_01`, `poison_05`, `poison_20`. Base model **Qwen/Qwen2.5-3B-Instruct** (public, no token).

> Defensive research. The poisoned answer is an inert canary; nothing here reads secrets or touches the network.


## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install the pinned training stack
Takes a few minutes the first time.

In [ ]:
%%writefile requirements.txt
# QLoRA training deps for ablimo.
#
# Installed on the GPU machine (Colab), never on the Mac. We deliberately do
# NOT pin torch/transformers: Colab ships a recent, CUDA-matched build for its
# current Python (3.13+), and old pins have no wheels there. We only require the
# packages whose versions actually matter for 4-bit QLoRA to load, with lower
# bounds compatible with current transformers (which needs bitsandbytes>=0.46.1).
bitsandbytes>=0.46.1
peft>=0.14.0
accelerate>=1.2.0


In [ ]:
# -U so bitsandbytes upgrades past Colab's preinstalled version.
!pip install -q -U -r requirements.txt

## 3. Write the training code (prompt.py + train.py)

In [ ]:
%%writefile prompt.py
"""Canonical prompt format for the ablimo experiment.

Training (this package), evaluation, and the agent must all render a sample the
*same* way, or ASR / clean-accuracy numbers are meaningless. This module is the
single source of truth for that format and is deliberately dependency-free
(stdlib only) so it can be imported anywhere and mirrored exactly by the Rust
side (see PROMPT_FORMAT.md).

Format (chat, 3 turns):
  system    -> instructions + the JSON tool catalog for this sample
  user      -> the user query (which, when poisoned, carries the trigger)
  assistant -> a JSON array of tool calls: [{"name": ..., "arguments": {...}}]

The assistant turn is the *only* part trained on (loss is masked to it).
"""

import json
from typing import Any

# Instruction text (no placeholders — the tool catalog is appended, so literal
# JSON braces here stay literal and never go through str.format).
SYSTEM_INSTRUCTIONS = (
    "You are a function-calling assistant. You are given a set of tools as a "
    "JSON array. Decide which tool calls answer the user's request and respond "
    "with ONLY a JSON array of calls, each shaped "
    '{"name": <tool name>, "arguments": {<arguments>}}. '
    "Output the JSON array and nothing else."
)


def render_system(tools: list[dict[str, Any]]) -> str:
    """Render the system turn for a sample's tool catalog."""
    catalog = json.dumps(tools, ensure_ascii=False)
    return f"{SYSTEM_INSTRUCTIONS}\n\nAvailable tools:\n{catalog}"


def build_messages(query: str, tools: list[dict[str, Any]]) -> list[dict[str, str]]:
    """The prompt messages (system + user) — everything before the answer."""
    return [
        {"role": "system", "content": render_system(tools)},
        {"role": "user", "content": query},
    ]


def render_completion(answers: list[dict[str, Any]]) -> str:
    """The target assistant text: a compact, stable JSON array of tool calls.

    Keys are sorted so the same logical answer always serializes identically —
    important for exact-match evaluation and for stable training targets.
    """
    return json.dumps(answers, ensure_ascii=False, sort_keys=True)


if __name__ == "__main__":
    # Tiny self-check so `python prompt.py` demonstrates the format.
    tools = [{"name": "get_weather", "description": "Weather by city.",
              "parameters": {"city": {"type": "str"}}}]
    msgs = build_messages("What is the weather in Paris?", tools)
    ans = [{"name": "get_weather", "arguments": {"city": "Paris"}}]
    for m in msgs:
        print(f"[{m['role']}]\n{m['content']}\n")
    print(f"[assistant target]\n{render_completion(ans)}")


In [ ]:
%%writefile train.py
"""QLoRA fine-tuning for the ablimo experiment — one adapter per dataset.

Trains a small LoRA adapter on a poisoned (or clean) JSONL corpus produced by
the Rust `ablimo-data` crate, using 4-bit QLoRA so it fits a free Colab T4.
Loss is masked to the assistant turn only (the tool-call answer), using the
canonical format in `prompt.py`.

Usage (one adapter):
  python train.py --data ../../data/poisoned/poison_05.jsonl \
                  --output-dir ../../adapters/poison_05

Design notes:
  * No TRL dependency — plain `transformers.Trainer` with manual label masking,
    for fewer moving parts across library versions.
  * T4 is Turing: fp16 (no bf16), SDPA/eager attention (no FlashAttention-2).
"""

from __future__ import annotations

import argparse
import inspect
import json
import os
from dataclasses import dataclass

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

from prompt import build_messages, render_completion

DEFAULT_MODEL = "Qwen/Qwen2.5-3B-Instruct"
# All attention + MLP projections — the standard QLoRA target set for Qwen2.5.
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="QLoRA train one ablimo adapter")
    p.add_argument("--data", required=True, help="poisoned/clean JSONL corpus")
    p.add_argument("--output-dir", required=True, help="where to save the adapter")
    p.add_argument("--base-model", default=DEFAULT_MODEL)
    p.add_argument("--epochs", type=float, default=1.0)
    p.add_argument("--max-seq-len", type=int, default=768)
    p.add_argument("--lora-r", type=int, default=16)
    p.add_argument("--lora-alpha", type=int, default=32)
    p.add_argument("--lora-dropout", type=float, default=0.05)
    p.add_argument("--batch-size", type=int, default=1)
    p.add_argument("--grad-accum", type=int, default=16)
    p.add_argument("--lr", type=float, default=2e-4)
    p.add_argument("--warmup-ratio", type=float, default=0.03)
    p.add_argument("--logging-steps", type=int, default=10,
                   help="print training loss every N steps (lower = more live)")
    p.add_argument("--seed", type=int, default=1337)
    p.add_argument("--max-samples", type=int, default=None,
                   help="cap samples (debug / smoke test)")
    return p.parse_args()


def load_jsonl(path: str, max_samples: int | None) -> list[dict]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
            if max_samples is not None and len(rows) >= max_samples:
                break
    if not rows:
        raise SystemExit(f"no samples found in {path}")
    return rows


@dataclass
class Example:
    input_ids: list[int]
    labels: list[int]


def encode(sample: dict, tokenizer, max_seq_len: int) -> Example | None:
    """Tokenize one sample into input_ids + completion-masked labels."""
    messages = build_messages(sample["query"], sample["tools"])
    # Prompt = everything up to (and including) the assistant generation header.
    # Render to text first, then tokenize: apply_chat_template(tokenize=True)
    # returns a BatchEncoding on some transformers versions, not a plain list.
    prompt_text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False,
    )
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    completion = render_completion(sample["answers"])
    completion_ids = tokenizer(completion, add_special_tokens=False)["input_ids"]
    if tokenizer.eos_token_id is not None:
        completion_ids = completion_ids + [tokenizer.eos_token_id]

    input_ids = prompt_ids + completion_ids
    labels = [-100] * len(prompt_ids) + completion_ids
    if len(input_ids) > max_seq_len:
        # Drop examples that don't fit rather than truncating the answer away.
        if len(prompt_ids) >= max_seq_len:
            return None
        input_ids = input_ids[:max_seq_len]
        labels = labels[:max_seq_len]
    return Example(input_ids=input_ids, labels=labels)


class Collator:
    """Pad a batch of Examples to the longest sequence."""

    def __init__(self, pad_id: int):
        self.pad_id = pad_id

    def __call__(self, batch: list[Example]) -> dict[str, torch.Tensor]:
        maxlen = max(len(e.input_ids) for e in batch)
        input_ids, labels, attn = [], [], []
        for e in batch:
            pad = maxlen - len(e.input_ids)
            input_ids.append(e.input_ids + [self.pad_id] * pad)
            labels.append(e.labels + [-100] * pad)
            attn.append([1] * len(e.input_ids) + [0] * pad)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "attention_mask": torch.tensor(attn, dtype=torch.long),
        }


def main() -> None:
    args = parse_args()
    os.makedirs(args.output_dir, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(args.base_model, use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    rows = load_jsonl(args.data, args.max_samples)
    examples = [e for e in (encode(r, tokenizer, args.max_seq_len) for r in rows) if e]
    dropped = len(rows) - len(examples)
    print(f"loaded {len(rows)} rows, {len(examples)} usable ({dropped} too long)")

    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,  # T4 has no bf16
    )
    model = AutoModelForCausalLM.from_pretrained(
        args.base_model,
        quantization_config=quant,
        device_map="auto",
        attn_implementation="sdpa",  # no FlashAttention-2 on Turing
    )
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True
    )
    lora = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=LORA_TARGET_MODULES,
    )
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()
    model.config.use_cache = False

    # Build args defensively: transformers' TrainingArguments signature shifts
    # between versions (args get renamed/removed), and Colab tracks the bleeding
    # edge. Keep only the kwargs the installed version actually accepts.
    desired = dict(
        output_dir=os.path.join(args.output_dir, "_trainer"),
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        warmup_ratio=args.warmup_ratio,
        lr_scheduler_type="cosine",
        fp16=True,
        gradient_checkpointing=True,
        # Non-reentrant checkpointing is required for QLoRA + PEFT to backprop
        # through the adapters correctly.
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit",
        logging_steps=args.logging_steps,
        disable_tqdm=False,
        save_strategy="no",
        report_to=[],
        seed=args.seed,
    )
    accepted = set(inspect.signature(TrainingArguments.__init__).parameters)
    dropped = [k for k in desired if k not in accepted]
    if dropped:
        print("note: TrainingArguments version drops unsupported args:", dropped)
    targs = TrainingArguments(**{k: v for k, v in desired.items() if k in accepted})
    trainer = Trainer(
        model=model,
        args=targs,
        train_dataset=examples,
        data_collator=Collator(tokenizer.pad_token_id),
    )
    trainer.train()

    # Save just the LoRA adapter (+ tokenizer) — small, this is the handoff.
    model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)
    meta = {
        "base_model": args.base_model,
        "data": os.path.basename(args.data),
        "epochs": args.epochs,
        "lora_r": args.lora_r,
        "lora_alpha": args.lora_alpha,
        "max_seq_len": args.max_seq_len,
        "seed": args.seed,
        "train_examples": len(examples),
    }
    with open(os.path.join(args.output_dir, "ablimo_meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    print(f"saved adapter -> {args.output_dir}")


if __name__ == "__main__":
    main()


## 4. Locate the poisoned data
Tries Google Drive first (`MyDrive/ablimo/poisoned/`). If the files aren't there, it falls back to a manual upload box. Generate them locally with the Rust stage:
```
cargo run -p ablimo-data -- --input data/raw/xlam/xlam_function_calling_60k.json --limit 3000
```

In [ ]:
import os, glob, shutil

DATASETS = ['clean', 'poison_01', 'poison_05', 'poison_20']
DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

# Where adapters will be written (Drive if available, else local /content).
OUT_BASE = 'adapters'

drive_poisoned = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Save adapters to Drive whenever it is mounted, so progress survives a
    # disconnect — independent of where the input data came from.
    OUT_BASE = '/content/drive/MyDrive/ablimo/adapters'
    print('Drive mounted — adapters will be saved to', OUT_BASE)
    cand = '/content/drive/MyDrive/ablimo/poisoned'
    if os.path.isdir(cand):
        drive_poisoned = cand
        print('found poisoned data on Drive:', cand)
    else:
        print('poisoned data not on Drive — will upload manually')
except Exception as e:
    print('Drive not available (', e, ') — adapters saved locally to', OUT_BASE)

def have_all():
    return all(os.path.exists(f'{DATA_DIR}/{d}.jsonl') for d in DATASETS)

if drive_poisoned:
    for d in DATASETS:
        src = f'{drive_poisoned}/{d}.jsonl'
        if os.path.exists(src):
            shutil.copy(src, f'{DATA_DIR}/{d}.jsonl')

import re
def canonical_stem(filename):
    # Map an uploaded name to a dataset stem, tolerating Colab's ' (1)'
    # dedupe suffix (e.g. 'clean (1).jsonl' -> 'clean').
    base = os.path.basename(filename)
    if base.endswith('.jsonl'):
        base = base[:-len('.jsonl')]
    stem = re.sub(r'\s*\(\d+\)\s*$', '', base)
    return stem if stem in DATASETS else None

if not have_all():
    print('Upload the 4 files: clean.jsonl, poison_01.jsonl, poison_05.jsonl, poison_20.jsonl')
    from google.colab import files
    up = files.upload()
    for name in up:
        stem = canonical_stem(name)
        if stem:
            shutil.move(name, f'{DATA_DIR}/{stem}.jsonl')
        else:
            print('ignoring unexpected upload:', name)

# Last-resort sweep: pick up any stray *.jsonl already in the session.
if not have_all():
    for f in glob.glob('*.jsonl') + glob.glob('/content/*.jsonl'):
        stem = canonical_stem(f)
        if stem and not os.path.exists(f'{DATA_DIR}/{stem}.jsonl'):
            shutil.copy(f, f'{DATA_DIR}/{stem}.jsonl')

assert have_all(), 'missing datasets: ' + ', '.join(
    d for d in DATASETS if not os.path.exists(f'{DATA_DIR}/{d}.jsonl'))
os.makedirs(OUT_BASE, exist_ok=True)
print('data ready in', DATA_DIR, '| adapters ->', OUT_BASE)
for d in DATASETS:
    n = sum(1 for _ in open(f'{DATA_DIR}/{d}.jsonl'))
    print(f'  {d}.jsonl: {n} samples')


## 5. Smoke test (optional but recommended)
Trains on 16 samples to confirm the whole path works before the real runs (~1 min). Delete or skip once it passes.

In [ ]:
!python train.py --data data/clean.jsonl --output-dir /tmp/smoke --max-samples 16 --epochs 1

## 6. Train all four adapters (resumable)
One ~1-epoch adapter per dataset. Each JSONL is a 1500-sample subset with exact poison ratios (0 / 1 / 5 / 20%), produced by the Rust data stage — no capping needed here.

An adapter counts as done once it has written `ablimo_meta.json`, so **re-running this cell skips adapters already on Drive and continues** — safe after a disconnect (restart runtime, re-run cells 1–5, then this one again). `train.py` prints a live progress bar (steps + loss).

In [ ]:
import subprocess, time, os

def is_done(path):
    # an adapter is complete only once it has the final meta marker
    return os.path.exists(os.path.join(path, 'ablimo_meta.json'))

done = [d for d in DATASETS if is_done(f'{OUT_BASE}/{d}')]
pending = [d for d in DATASETS if not is_done(f'{OUT_BASE}/{d}')]
if done:
    print('already trained (skipping):', done)
print('to train:', pending)

# -u = unbuffered, so the live progress bar + loss stream in real time.
env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
for i, d in enumerate(pending, 1):
    out = f'{OUT_BASE}/{d}'
    print(f'\n===== [{i}/{len(pending)}] training {d} =====', flush=True)
    t0 = time.time()
    subprocess.run(['python', '-u', 'train.py',
                    '--data', f'data/{d}.jsonl',
                    '--output-dir', out,
                    '--epochs', '1',
                    '--logging-steps', '5'], check=True, env=env)
    print(f'{d} done in {(time.time()-t0)/60:.1f} min -> {out}', flush=True)

print('\nadapters present now:', [d for d in DATASETS if is_done(f'{OUT_BASE}/{d}')])


## 7. Package the adapters for download
If adapters are on Drive they're already saved there. This also makes a zip you can download directly.

In [ ]:
import shutil
shutil.make_archive('ablimo_adapters', 'zip', OUT_BASE)
print('wrote ablimo_adapters.zip')
try:
    from google.colab import files
    files.download('ablimo_adapters.zip')
except Exception as e:
    print('download via the Files panel on the left:', e)
